In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
# Load the dataset 



In [ ]:
data = pd.read_csv('ventas_ml_clase2.csv')
display(data.head())


In [ ]:
print("""
Filas: {}, Columnas: {}
""".format(data.shape[0], data.shape[1]))

print("""Tipos de datos:
{}
""".format(data.dtypes))    

In [ ]:
data.describe(include='all').transpose().head()

In [ ]:
# Predecir las ventaas usando vaiables precio y temporada, tienda, canal
X = data[['marketing', 'precio', 'temporada', 'tienda', 'canal']]  # Features
y = data['ventas']  # Target variable

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Tamaño del conjunto de entrenamiento: {}, Tamaño del conjunto de prueba: {} \n Porcentaje de prueba: {:.2f}%".format(X_train.shape[0], X_test.shape[0], (X_test.shape[0] / X.shape[0]) * 100))


# Define the preprocessing for categorical features
categorical_features = ['temporada', 'tienda', 'canal']
categorical_transformer = OneHotEncoder(drop='first')
# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'  # Keep the 'precio' column as is
)
# Create a pipeline that combines preprocessing and the regression model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
# Train the model
pipeline.fit(X_train, y_train)
# Make predictions
y_pred = pipeline.predict(X_test)
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Squared Error: {mse:.2f}')
print(f'R^2 Score: {r2:.2f}')
print(f'Mean Absolute Error: {mae:.2f}')
print(f"="*50)
print(f"Media de las ventas reales: {y_test.mean():.2f}")
print(f"Media de las ventas predichas: {y_pred.mean():.2f}")
print(f"Error relativo (MAE / Media de las ventas reales): {mae / y_test.mean():.2%}")
print(f"="*50)
print("\nCoeficientes del modelo:")
print("Intercepto:", pipeline.named_steps['regressor'].intercept_)
print("Coeficientes:", pipeline.named_steps['regressor'].coef_)
print(f"="*50)
print(f"Interpretaión de las metricas:")
print(f"- El Mean Squared Error (MSE) de {mse:.2f} indica que, en promedio, las predicciones del modelo se desvían de los valores reales en aproximadamente {mse:.2f} unidades cuadradas. Un MSE más bajo indica un mejor ajuste del modelo a los datos.")
print(f"- El R^2 Score de {r2:.2f} sugiere que el modelo explica aproximadamente el {r2:.2%} de la variabilidad en las ventas. Un R^2 cercano a 1 indica un modelo que se ajusta bien a los datos, mientras que un R^2 cercano a 0 indica un modelo que no explica la variabilidad de los datos.")
print(f"- El Mean Absolute Error (MAE) de {mae:.2f} indica que, en promedio, las predicciones del modelo se desvían de los valores reales en aproximadamente {mae:.2f} unidades. Un MAE más bajo indica un mejor ajuste del modelo a los datos.")
print(f"el {(1-r2)*100:.2f}% de la variabilidad en las ventas no es explicada por el modelo, lo que sugiere que hay otros factores no incluidos en el modelo que podrían estar influyendo en las ventas.")

if r2 >= 0.8:
    print("Valoracion: buen ajuste para un modelo lineal")
elif r2 >= 0.5:
    print("Valoracion: ajuste moderado para un modelo lineal")
else:
    print("Valoracion: ajuste pobre para un modelo lineal")

pipeline


In [ ]:

# draw the regression line 
import matplotlib.pyplot as plt
import seaborn as sns

# Create a scatter plot of actual vs predicted values
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', label='Perfect Prediction')  # Add a reference line for perfect predictions
# Limit the axes to the range of actual values for better visualization
plt.xlim(y.min(), y.max())
plt.ylim(y.min(), y.max())
# set median and mean lines
plt.axhline(y=y_test.mean(), color='blue', linestyle='--', label='Mean Actual Sales')
plt.axhline(y=y_pred.mean(), color='green', linestyle='--', label='Mean Predicted Sales')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Actual vs Predicted Sales')
plt.legend()
plt.show()

# Coeficientes del modelo: ¿que variables pesan más?

Un modelo de regresion lineal es una formula de  ***Sumas ponderadas***

ventas = base + (peso1 x marketing) + (peso2 * precio) + (peso3 * temporada) + ....



In [ ]:
# Extract los coeficientes (peso) que el modelo aprendió para cada variable
coeficientes = pipeline.named_steps['regressor'].coef_
print("\nCoeficientes del modelo:")
print("Intercepto:", pipeline.named_steps['regressor'].intercept_)

# Obtener los nombres de las características después del OneHotEncoding
ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
feature_names = ohe.get_feature_names_out(categorical_features)
print("Nombres de las características después del OneHotEncoding:")
print(feature_names)

# Los coeficientes corresponden a las características en el siguiente orden:
# ['temporada_primavera', 'temporada_verano', 'temporada_otono', 'tienda_Tienda B', 'tienda_Tienda C', 'canal_Online', 'precio']
print("Coeficientes correspondientes a cada característica:")
for feature, coef in zip(feature_names, coeficientes[:-1]):
    print(f"{feature}: {coef:.4f}")


In [ ]:
row = X_test.iloc[[0]].copy()  # Selecciona la primera fila del conjunto de prueba
pred_base = float(pipeline.predict(row)[0])
row_more_marketing = row.copy()
row_more_marketing['marketing'] += float(row_more_marketing['marketing'].values[0]) * 1.1

pred_more_marketing = float(pipeline.predict(row_more_marketing)[0])  # Incrementa el gasto en marketing en un 10% y predice nuevamente

# Calcular la diferencia en la predicción
mkt_original = row['marketing'].values[0]
mkt_nuevo = row_more_marketing['marketing'].values[0]
diferencia_prediccion = pred_more_marketing - pred_base
print(f"\nPredicción original: {pred_base:.2f}")
print(f"Predicción con marketing incrementado: {pred_more_marketing:.2f}")
print(f"Diferencia en la predicción: {diferencia_prediccion:.2f}")
print(f"\nEfecto del incremento en marketing: {diferencia_prediccion:.2f} unidades de ventas adicionales por cada {mkt_nuevo - mkt_original:.2f} unidades de incremento en marketing.")

In [ ]:
row = X_test.iloc[[0]].copy()  # Selecciona la primera fila del conjunto de prueba
print("\nFila seleccionada para la predicción:")
print(row)
prediccion = pipeline.predict(row)
print(f"\nPredicción para la fila seleccionada: {prediccion[0]:.2f}")